# 🧑‍💻🤖 AI Codebase Understanding and Modernization Agent
**Portfolio project | Python | Static analysis | Code search | Modernization suggestions**

A local codebase assistant that scans a repository, indexes source files, retrieves relevant code snippets, summarizes modules, and surfaces conservative modernization suggestions.

### Features
- 📁 Scan a repository while skipping common generated folders.
- 🔎 Search source files using TF-IDF similarity.
- 🧩 Extract Python classes, functions, and imports without executing code.
- 📝 Generate heuristic module summaries.
- 🛡️ Flag potentially risky patterns for human review.
- 🧪 Run basic validation checks.

> **Safety:** Read-only prototype. It does not rewrite files or execute repository code. Review suggestions before changing a codebase.

## 1. Install dependencies
Run in Jupyter or Google Colab.

In [ ]:
%pip -q install pandas scikit-learn

## 2. Imports and configuration
Set `REPO_PATH` to a repository directory available in your environment.

In [ ]:
from pathlib import Path
import ast
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

REPO_PATH = Path(".").resolve()  # Change to your repository path.
ALLOWED_EXTENSIONS = {".py", ".js", ".jsx", ".ts", ".tsx", ".java", ".go", ".rs", ".cpp", ".c", ".h", ".hpp", ".cs", ".php", ".rb", ".swift", ".kt", ".sql", ".sh", ".yaml", ".yml", ".toml", ".json", ".md"}
SKIP_DIRS = {".git", ".venv", "venv", "node_modules", "__pycache__", "dist", "build", ".next", "coverage", ".idea", ".vscode"}
MAX_FILE_BYTES = 500_000
print("Repository path:", REPO_PATH, "| Exists:", REPO_PATH.exists())

## 3. Scan the repository safely
Reads selected file types, skips generated folders, and ignores very large files.

In [ ]:
def scan_repository(repo_path, max_file_bytes=MAX_FILE_BYTES):
    repo_path = Path(repo_path).resolve()
    if not repo_path.exists() or not repo_path.is_dir():
        raise ValueError(f"Repository path is not a directory: {repo_path}")
    records = []
    for path in repo_path.rglob("*"):
        if not path.is_file() or any(part in SKIP_DIRS for part in path.parts):
            continue
        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue
        try:
            if path.stat().st_size > max_file_bytes:
                continue
            content = path.read_text(encoding="utf-8", errors="replace")
        except OSError:
            continue
        if content.strip():
            records.append({"path": str(path.relative_to(repo_path)), "extension": path.suffix.lower(),
                            "characters": len(content), "content": content})
    return pd.DataFrame(records, columns=["path", "extension", "characters", "content"])

files_df = scan_repository(REPO_PATH)
print(f"Indexed {len(files_df)} files.")
files_df[["path", "extension", "characters"]].head(20)

## 4. Build a code search index
TF-IDF is a simple offline baseline. Larger repositories may need chunking and embeddings.

In [ ]:
if len(files_df):
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=50_000)
    code_matrix = vectorizer.fit_transform((files_df["path"] + " " + files_df["content"]).tolist())
else:
    vectorizer, code_matrix = None, None

def search_code(query, top_k=5):
    if vectorizer is None or code_matrix is None:
        return pd.DataFrame(columns=["path", "score", "snippet"])
    scores = cosine_similarity(vectorizer.transform([query]), code_matrix).ravel()
    rows = []
    for idx in scores.argsort()[::-1][:top_k]:
        row = files_df.iloc[idx]
        rows.append({"path": row["path"], "score": round(float(scores[idx]), 4), "snippet": row["content"][:1200]})
    return pd.DataFrame(rows)

search_code("authentication login password validation")

## 5. Extract Python structure
Use `ast` to identify classes, functions, and imports without executing Python files.

In [ ]:
def analyze_python_file(source):
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return {"classes": [], "functions": [], "imports": [], "parse_error": f"SyntaxError at line {exc.lineno}: {exc.msg}"}
    classes, functions, imports = [], [], []
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            classes.append({"name": node.name, "line": node.lineno})
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append({"name": node.name, "line": node.lineno, "async": isinstance(node, ast.AsyncFunctionDef)})
        elif isinstance(node, ast.Import):
            imports.extend(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imports.append(node.module or "")
    return {"classes": classes, "functions": functions, "imports": sorted(set(x for x in imports if x)), "parse_error": None}

def build_structure_report(files):
    reports = []
    for _, row in files.iterrows():
        if row["extension"] == ".py":
            parsed = analyze_python_file(row["content"])
            reports.append({"path": row["path"], "class_count": len(parsed["classes"]),
                            "function_count": len(parsed["functions"]), "imports": ", ".join(parsed["imports"][:12]),
                            "parse_error": parsed["parse_error"] or ""})
    return pd.DataFrame(reports)

structure_df = build_structure_report(files_df)
structure_df.head(20)

## 6. Generate heuristic module summaries
These are extractive summaries, not LLM-generated explanations.

In [ ]:
def summarize_module(path, content, max_lines=8):
    nonempty = [line.strip() for line in content.splitlines() if line.strip()]
    meaningful = [line for line in nonempty if line.startswith(("#", "class ", "def ", "async def ", "function ", "export ", "import ", "from "))]
    return {"path": path, "summary": " | ".join((meaningful or nonempty)[:max_lines])[:1000],
            "line_count": len(content.splitlines())}

summary_df = pd.DataFrame([summarize_module(r["path"], r["content"]) for _, r in files_df.iterrows()])
summary_df.head(20)

## 7. Flag patterns for human review
These text matches are not proof of vulnerabilities and may produce false positives.

In [ ]:
RISK_PATTERNS = {
    "possible_hardcoded_secret": re.compile(r"(?i)(api[_-]?key|secret|password|token)\s*[:=]\s*['\"][^'\"]{8,}['\"]"),
    "dynamic_eval": re.compile(r"\beval\s*\("),
    "shell_execution": re.compile(r"\b(subprocess\.(run|Popen|call)|os\.system)\s*\("),
    "broad_exception": re.compile(r"except\s*:")
}

def find_review_flags(files):
    findings = []
    for _, row in files.iterrows():
        for line_no, line in enumerate(row["content"].splitlines(), start=1):
            for rule, pattern in RISK_PATTERNS.items():
                if pattern.search(line):
                    findings.append({"path": row["path"], "line": line_no, "rule": rule,
                                     "note": "Review manually; this may be a false positive."})
    return pd.DataFrame(findings, columns=["path", "line", "rule", "note"])

review_flags_df = find_review_flags(files_df)
review_flags_df.head(30)

## 8. Ask a codebase question
This retrieves relevant snippets; similarity does not guarantee correctness.

In [ ]:
def answer_codebase_question(question, top_k=3):
    results = search_code(question, top_k)
    if results.empty:
        return "No indexed files found. Set REPO_PATH to a repository with supported source files."
    sections = ["## Retrieved code context", "Question: " + question]
    for _, row in results.iterrows():
        sections.append(f"### {row['path']} (similarity: {row['score']})\n```\n{row['snippet']}\n```")
    sections.append("Review the retrieved context before making changes. Similarity is not a correctness guarantee.")
    return "\n\n".join(sections)

print(answer_codebase_question("Where is user authentication handled?", top_k=3))

## 9. Basic checks
These checks validate baseline parsing behavior; they do not certify code quality or security.

In [ ]:
example = '''
import os
class Example:
    def run(self):
        return True
def helper():
    pass
'''
parsed = analyze_python_file(example)
assert any(c["name"] == "Example" for c in parsed["classes"])
assert any(f["name"] == "helper" for f in parsed["functions"])
assert "os" in parsed["imports"]
assert analyze_python_file("def broken(:")["parse_error"] is not None
print("All notebook baseline checks passed.")

## 10. Suggested portfolio upgrades
- 🧠 Add LLM-generated explanations with file and line citations.
- 🧩 Chunk large files by function/class.
- 🔎 Add embeddings, vector search, and reranking.
- 🗺️ Build a dependency graph and module relationship explorer.
- 🛠️ Generate proposed diffs for human approval, never automatic rewriting.
- 🧪 Run tests in an isolated sandbox and compare results.
- 🔐 Add secret scanning, dependency auditing, and prompt-injection defenses.
- 🌐 Build a FastAPI backend and interactive UI.
- 📈 Evaluate retrieval quality, usefulness, and latency.

### Responsible use
Treat repository files as untrusted input. Do not execute unknown code, expose secrets in logs, or send proprietary source code to external APIs without permission. Keep this initial tool read-only and require human approval for modifications.